# Why systems push back {#sec-push-back}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/book-ai/notebooks/c01-push-back.ipynb)

A hospital group's outpatient waiting list has more than doubled in ten years. Each year brought extra clinics, a validation drive or a funded initiative, and each time the list dipped, stalled or kept on climbing. The people running the service are not careless. They are working inside a system that answers every push with a push of its own. This chapter shows you how to see that system as **stocks**, **flows** and **feedback**, and how to put it into a model you can run in Python.

**Who it is for.** Anyone starting system dynamics, whether or not you have written much Python. Every line of code is explained, and @sec-python teaches the Python the book uses if you want it first.

**Who can skip what.** If you already build stock-and-flow models, skim sections 2 and 6 and read sections 3 to 5, which set up the Python habits the rest of the book relies on. If you write Python every day, skim the code explanations in section 3.

**Prerequisites.** None beyond school algebra. You need a Python environment with numpy, pandas and matplotlib, or the Colab button above.

**Time.** About three hours, including the exercises.

The figure below is the whole of this chapter's first model: one stock, the patients waiting, filled by referrals and drained by treatments. It is drawn by `draw_stock_flow`, a helper after the setup cell that places matplotlib shapes (`Rectangle`, `Ellipse`, `Polygon`) with `ax.add_patch` and draws arrows with `ax.annotate`; you can treat it as a black box.

In [ ]:
# Setup: run this cell first. In Colab it installs loopbricks (ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/"
                           "loopbricks.git@book-ai#subdirectory=python"])
    import loopbricks as lb
import os, pandas as pd
# Use the local copy of the data if there is one, otherwise read it from GitHub.
DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/book-ai/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])
import copy                            # copies nested dictionaries (section 6)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cycler
from matplotlib.patches import Ellipse, Polygon, Rectangle

SEED = 1                               # one seed per chapter
rng = np.random.default_rng(SEED)      # (this chapter draws no random numbers)
# Colour-blind-safe colours (Okabe and Ito) with a different line style
# for each series, so every figure also reads in greyscale.
OKABE_ITO = ["#000000", "#E69F00", "#0072B2", "#009E73", "#D55E00",
             "#CC79A7", "#56B4E9"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1)),
          (0, (1, 3))]
# plt.rcParams holds matplotlib's defaults. Adding two cyclers pairs the
# first colour with the first line style, the second with the second, ...
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "figure.dpi": 100})

In [ ]:
def draw_stock_flow(ax, stock, inflow, outflow, inputs_in=(),
                    inputs_out=(), title=""):
    """Draw one stock with an inflow and an outflow.

    Clouds are the world outside the model, thick arrows are pipes, bow ties
    are valves (the flows) and the box is the stock. Thin grey arrows carry
    information: what each flow is worked out from.
    """
    ax.set_xlim(0, 10)
    ax.set_ylim(0.6, 4.8)
    ax.axis("off")
    y = 2.0
    # the stock: a box with its lower-left corner at (4, y - 0.6)
    ax.add_patch(Rectangle((4, y - 0.6), 2, 1.2, fc="white", ec="black",
                           lw=1.5, zorder=2))
    ax.text(5, y, stock, ha="center", va="center", fontsize=11, zorder=3)
    for x in (0.7, 9.3):                     # a cloud at each end
        ax.add_patch(Ellipse((x, y), 1.0, 0.7, fc="white", ec="grey",
                             ls="--"))
    # pipes: annotate("", xy=end, xytext=start) draws an arrow with no text
    pipe = dict(arrowstyle="-|>,head_width=0.4,head_length=0.6", lw=3,
                color="black")
    ax.annotate("", xy=(4, y), xytext=(1.2, y), arrowprops=pipe)
    ax.annotate("", xy=(8.8, y), xytext=(6, y), arrowprops=pipe)
    # information links: thin arrows, bent into an arc by rad
    info = dict(arrowstyle="->", lw=0.9, color="grey",
                connectionstyle="arc3,rad=0.15")
    for x, name, inputs in ((2.6, inflow, inputs_in),
                            (7.4, outflow, inputs_out)):
        # the valve: a bow tie drawn as a Polygon through four corners
        bow = [(x - 0.25, y + 0.35), (x + 0.25, y - 0.35),
               (x + 0.25, y + 0.35), (x - 0.25, y - 0.35)]
        ax.add_patch(Polygon(bow, fc="white", ec="black", zorder=3))
        ax.text(x, y - 0.5, name, ha="center", va="top", fontsize=10)
        for j, label in enumerate(inputs):
            lx = x + (j - (len(inputs) - 1) / 2) * 2.2
            ax.text(lx, 4.3, label, ha="center", va="center", fontsize=9)
            ax.annotate("", xy=(x, y + 0.4), xytext=(lx, 4.05),
                        arrowprops=info)
    # the outflow also depends on the stock itself
    ax.annotate("", xy=(7.2, y + 0.3), xytext=(5.8, y + 0.6),
                arrowprops=dict(info, connectionstyle="arc3,rad=-0.5"))
    ax.set_title(title)


fig, ax = plt.subplots(figsize=(9, 3.4))
draw_stock_flow(ax, "Waiting list\n(patients)", "Referrals", "Treatments",
                inputs_in=["Referral rate"],
                inputs_out=["Capacity", "Minimum treatment time"],
                title="The waiting list as a stock and two flows")
plt.show()

Figure: stock-and-flow diagram of the waiting list. The box is the stock of patients waiting; referrals flow in and treatments flow out through the valves (bow ties). Clouds mark where patients come from and go to outside the model. Grey arrows show what each flow depends on: treatments depend on capacity, on a minimum treatment time and on the list itself.

## Learning outcomes

By the end of this chapter you will be able to:

1. **Read** a stock series for the net flow it implies, and explain why the separate flows cannot be recovered from it (Try it 2).
2. **Build** a one-stock model in plain Python with Euler's method, and **explain** each line (Try it 3).
3. **Reproduce** the same model with the `loopbricks` package and **verify** that the two agree to the last digit (Try it 4).
4. **Choose** a time step and **justify** it with a halving test (Try it 5).
5. **Diagnose** policy resistance by comparing a policy run with and without a feedback loop (Try it 6).
6. **Use** the Bathtub, Conversion, Split Flow and Broken Cascade molecules and **say** what each one is for (Try it 7).

## 1. A list that keeps growing

The data for the running case is a monthly count of patients waiting for a first outpatient appointment in one hospital group, January 2014 to December 2023, split by how long they have waited. It is synthetic, shaped like published Irish waiting-list counts; chapter 13 explains how it was made. The code below plots the total.

In [ ]:
total = wl.set_index("month")["total"]       # patients waiting, by month

fig, ax = plt.subplots()
ax.plot(total.index, total / 1000, label="Patients waiting")
start_extra = pd.Timestamp("2022-01-01")
ax.axvline(start_extra, color="grey", ls=":", lw=1)
# pd.Timedelta(days=40) is a length of time: start the label 40 days to the right
ax.text(start_extra + pd.Timedelta(days=40), 36,
        "extra activity\nbought from\nJanuary 2022", fontsize=9)
ax.set_title("Outpatient waiting list, one hospital group")
ax.set_xlabel("Month")
ax.set_ylabel("Patients waiting (thousands)")
ax.legend(loc="upper left")
plt.show()

# total["2014-01"] selects the rows of that month (here one); .iloc[0] takes its value
for month in ["2014-01", "2019-12", "2021-12", "2023-12"]:
    print(f"{month}: {total[month].iloc[0]:>6,} patients waiting")
print(f"Change, January 2014 to December 2023: "
      f"{total.iloc[-1] / total.iloc[0] - 1:+.0%}")
over_a_year = wl.set_index("month")["w12_plus"] / total
print(f"Share waiting over a year: {over_a_year.iloc[0]:.0%} in January "
      f"2014, {over_a_year.iloc[-1]:.0%} in December 2023")

Figure: patients waiting each month, January 2014 to December 2023 (synthetic data shaped like Irish outpatient counts). The dotted line marks the start of a funded initiative in January 2022.

The list rose from 32,276 patients in January 2014 to 43,839 at the end of 2019, jumped to 68,552 by the end of 2021 after activity collapsed in 2020, and then stopped rising at about 70,000. Over the decade it grew by 117%. The share of patients waiting more than a year rose from 29% to 50%. The extra activity bought from January 2022 did not bring the list down; it only stopped it growing.

That is the puzzle this book keeps returning to. Why did a decade of effort not shrink the list, and what would? Before we can answer, we need a way to think about anything that accumulates.

## 2. Stocks, flows and the bathtub

A **stock** is a quantity that accumulates: patients waiting, money in an account, water in a bath, staff on the payroll. You can count it at an instant. A **flow** is a rate that changes a stock: referrals a month, treatments a month. You can only measure it over a period. A stock changes only through its flows, so over any stretch of time

$$
\text{Waiting list}(t) = \text{Waiting list}(0) + \int_0^t \big(\text{Referrals}(s) - \text{Treatments}(s)\big)\,ds .
$$

Equivalently, the rate of change of the stock is the **net flow**, inflow minus outflow:

$$
\frac{d\,\text{Waiting list}}{dt} = \text{Referrals} - \text{Treatments}.
$$

This is the **bathtub** idea. The level in a bath tells you nothing about how fast the taps run, only about how much more has come in than has gone out. A level that is not moving can hide a torrent in and a torrent out. The data has the same property: it records the stock, the patients waiting, but no hospital in our case publishes referrals or treatments. The month-to-month change is all the data tells us about the flows.

In [ ]:
net = total.diff()                 # this month's list minus last month's
yearly = net.groupby(net.index.year).mean()

fig, ax = plt.subplots()
ax.bar(net.index, net, width=25, color="#0072B2")
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Monthly change in the waiting list (the net flow)")
ax.set_xlabel("Month")
ax.set_ylabel("Change (patients a month)")
plt.show()

print("Average net flow by year (patients a month):")
print(yearly.round(0).astype(int).to_string())   # astype(int): show whole numbers

Figure: month-to-month change in the waiting list, which is the net flow, referrals minus treatments (patients a month). Bars above zero are months when more patients joined than left.

From 2014 to 2016 the net flow averaged under 100 patients a month; from 2017 to 2019 it was between 213 and 288. In 2020 it averaged 1,596 a month, which is what happens to a bath when the plug is half closed. In 2023 it was 16 a month: almost exactly balanced. Notice how small these numbers are. Something like 4,000 patients join and leave the list each month, and a gap of 2% between them is enough to add 1,000 patients a year.

**Try it 2.**

- *Check.* In how many months of 2022 and 2023 did the list fall?
- *Apply.* Show that the average monthly net flow over the whole decade equals the last value minus the first, divided by the number of months between them. Check it with the data.
- *Extend.* Suppose referrals were 4,000 patients a month in 2014. What treatment rate does the 2014 average net flow imply? Could you find both flows if you knew neither?

<details><summary>Worked answer</summary>

*Check.* The list fell in 8 of the 24 months (the cell below counts them). The initiative balanced the flows; it did not reverse them.

*Apply.* The sum of the monthly changes telescopes: each month's value is added once and subtracted once, except the first and the last. So the mean change is (69,927 minus 32,276) / 119 = 316 patients a month, and `net.mean()` gives the same.

*Extend.* The 2014 average net flow was 96 a month, so treatments would be about 3,904 a month. Without referrals you cannot tell: 5,000 in and 4,904 out gives exactly the same stock. The stock pins down only the difference. Chapter 13, Data, calibration and uncertainty, deals with this by bringing in other evidence and by asking which flows a model with sensible structure would produce.
</details>

In [ ]:
recent = net["2022":"2023"]
print(f"Months in 2022 and 2023 when the list fell: {(recent < 0).sum()} "
      f"of {len(recent)}")
months_between = len(total) - 1
print(f"(last - first) / months: "
      f"{(total.iloc[-1] - total.iloc[0]) / months_between:.1f}")
print(f"mean of the monthly changes: {net.mean():.1f}")
print(f"Treatments implied in 2014 if referrals were 4,000: "
      f"{4000 - yearly[2014]:,.0f} a month")

## 3. A first model in plain Python

A model of the waiting list needs rules for the two flows. We start with the simplest rules that make sense, taken from the running case:

- **Referrals** arrive at a constant `Referral rate` of 4,000 patients a month.
- **Treatments** run at the `Capacity` of the clinics, 3,950 patients a month, unless the list is so short that there are not enough patients to fill the clinics. A patient needs at least a `Minimum treatment time` of 0.5 month to be booked and seen, so the list can supply at most `Waiting list / Minimum treatment time` patients a month:

$$
\text{Treatments} = \min\left(\text{Capacity},\ \frac{\text{Waiting list}}{\text{Minimum treatment time}}\right).
$$

The second rule stops the model treating patients who are not there, so the list can never go below zero for a physical reason, not because of a patch in the code.

To run the model, a computer cannot integrate the flows exactly. Instead it takes small **time steps** of length $\Delta t$ (the code calls it `dt`). At each step it works out the flows from the stock as it is now, assumes they stay the same for the whole step, and moves the stock by the net flow times the step:

$$
\text{Waiting list}(t + \Delta t) = \text{Waiting list}(t) + \Delta t\,\big(\text{Referrals}(t) - \text{Treatments}(t)\big).
$$

This is **Euler's method**. It is all a system dynamics simulator does underneath, and you can write it in a few lines. In the code below, a function (`def`) packages the steps so we can run them again with other numbers; `out` is a dictionary of lists that grows by one entry each step; `for i in range(steps + 1)` repeats the indented block once for each step.

In [ ]:
def simulate(T=119, dt=0.125, waiting_list=32000.0, referral_rate=4000.0,
             capacity=3950.0, minimum_treatment_time=0.5):
    """Run the waiting-list model with Euler's method.

    Time is in months, the stock in patients and the flows in patients a
    month. Returns a dictionary of lists with one entry per time step.
    """
    steps = round(T / dt)                    # number of steps of length dt
    out = {"time": [], "Waiting list": [], "Referrals": [], "Treatments": []}
    for i in range(steps + 1):
        t = i * dt
        # 1. Work out the flows from the stock as it is now.
        referrals = referral_rate
        treatments = min(capacity, waiting_list / minimum_treatment_time)
        # 2. Record this moment.
        out["time"].append(t)
        out["Waiting list"].append(waiting_list)
        out["Referrals"].append(referrals)
        out["Treatments"].append(treatments)
        if i == steps:                       # the last row: stop here
            break
        # 3. Euler's step: move the stock by dt times the net flow.
        waiting_list = waiting_list + dt * (referrals - treatments)
        waiting_list = max(waiting_list, 0.0)    # a list cannot be negative
    return out


mine = simulate()
print(f"Steps taken: {len(mine['time']) - 1}")
print(f"Waiting list after 119 months: {mine['Waiting list'][-1]:,.0f}")
print(f"Data in December 2023: {total.iloc[-1]:,}")

months = np.arange(len(total))               # 0 = January 2014
fig, ax = plt.subplots()
ax.plot(months, total.to_numpy() / 1000, label="Data")
ax.plot(mine["time"], np.array(mine["Waiting list"]) / 1000,
        label="Model: constant flows")
ax.set_title("A first model against the data")
ax.set_xlabel("Months since January 2014")
ax.set_ylabel("Patients waiting (thousands)")
ax.legend()
plt.show()

Figure: the first model (constant referrals of 4,000 and capacity of 3,950 patients a month) against the data, in months since January 2014.

The model takes 952 steps of an eighth of a month and ends at 37,950 patients, against 69,927 in the data. With a constant net flow of 50 patients a month the list rises in a straight line, 32,000 plus 50 times 119. That is the first rule of reading stocks: **a constant net flow gives a straight line; a net flow that changes gives a curve.** The data is not a straight line, so the flows are not constant. A model that matched would need some combination of growing referrals, falling capacity and a shock in 2020. Chapter 5, Loops that grow, settle and tip, adds growing referrals; chapter 13 fits the model to the data.

**Try it 3.**

- *Check.* Change one argument of `simulate` so that the list stays at 32,000 for the whole run.
- *Apply.* What constant capacity would bring the model to the December 2023 value of 69,927, with referrals at 4,000?
- *Extend.* Add a `capacity_drop` argument: from month 74 (March 2020) to month 83, capacity is 40% lower. How much does that shock alone add to the list?

<details><summary>Worked answer</summary>

*Check.* `simulate(capacity=4000)` makes referrals and treatments equal, and the list never moves from 32,000.

*Apply.* The list must rise by 37,927 in 119 months, 318.7 a month, so capacity must be 4,000 minus 318.7, or about 3,681 patients a month. The cell below confirms that the run ends at 69,927.

*Extend.* Nine months of a capacity 1,580 lower add 9 times 1,580, or 14,220 patients, and the list never wins them back because capacity afterwards is still below referrals. The run ends at 52,170 instead of 37,950. The real list grew by 24,713 from December 2019 to December 2021, so the shock explains a lot, but not all, of the jump.
</details>

In [ ]:
print(f"capacity 4,000: list ends at "
      f"{simulate(capacity=4000)['Waiting list'][-1]:,.0f}")
needed = 4000 - (69927 - 32000) / 119
print(f"capacity needed: {needed:,.1f}; list ends at "
      f"{simulate(capacity=needed)['Waiting list'][-1]:,.0f}")


def simulate_with_shock(T=119, dt=0.125, waiting_list=32000.0,
                        referral_rate=4000.0, capacity=3950.0,
                        minimum_treatment_time=0.5, capacity_drop=0.4):
    """As simulate(), but capacity is lower from month 74 to month 83."""
    steps = round(T / dt)
    for i in range(steps):
        t = i * dt
        # "a if condition else b" gives a when the condition holds, b otherwise
        cap = capacity * (1 - capacity_drop) if 74 <= t < 83 else capacity
        treatments = min(cap, waiting_list / minimum_treatment_time)
        waiting_list = max(waiting_list + dt * (referral_rate - treatments),
                           0.0)
    return waiting_list


print(f"with the 2020 shock: list ends at {simulate_with_shock():,.0f}")
print(f"real growth, Dec 2019 to Dec 2021: "
      f"{total['2021-12'].iloc[0] - total['2019-12'].iloc[0]:,}")

## 4. The same model with loopbricks

Writing the loop by hand teaches you what a simulator does. For anything bigger you want the model written down as data, so that it can be checked, shared, drawn and run in other ways. The `loopbricks` package does this. A model is a Python dictionary with three lists, `stocks`, `flows` and `vars` (auxiliary variables and constants). Each element has an `id`, a display `name`, an equation `eq` and its `units`. A flow names the stock it drains (`from`) and the stock it fills (`to`). In equations, names are written with underscores: `Waiting_list` is the stock called "Waiting list". Setting `nonNegative` says the stock is physical and must not go below zero; the [package README](https://github.com/waelrash1/loopbricks/tree/main/python) lists every option.

In [ ]:
case_spec = {
    "name": "Waiting list", "timeUnit": "month", "T": 119, "dt": 0.125,
    "stocks": [
        {"id": "wl", "name": "Waiting list", "init": "32000",
         "units": "patients", "nonNegative": True},
    ],
    "flows": [
        {"id": "ref", "name": "Referrals", "to": "wl",
         "eq": "Referral_rate", "units": "patients/month"},
        {"id": "trt", "name": "Treatments", "from": "wl",
         "eq": "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
         "units": "patients/month"},
    ],
    "vars": [
        {"id": "rr", "name": "Referral rate", "eq": "4000",
         "units": "patients/month"},
        {"id": "cap", "name": "Capacity", "eq": "3950",
         "units": "patients/month"},
        {"id": "mtt", "name": "Minimum treatment time", "eq": "0.5",
         "units": "month"},
    ],
}
case = lb.Model.from_spec(case_spec)
print(case.equations())
print()
print("Constants you can set:", case.params)

`equations()` prints the model in the integral form you met in section 2: `INTEG(inflow - outflow, initial value)`. `params` lists the constants, under the names you use to change them. Now run it and compare with our own loop.

In [ ]:
df = case.run()                         # a pandas DataFrame indexed by time
mine_list = np.array(mine["Waiting list"])
gap = np.abs(df["Waiting list"].to_numpy() - mine_list).max()
print(f"Rows: {len(df)}; largest difference from the plain Python loop: "
      f"{gap} patients")
print("Units:", df.attrs["units"])
df.loc[[0, 12, 60, 119], ["Waiting list", "Referrals", "Treatments"]]

The largest difference is exactly zero: the package does the same arithmetic in the same order as the fifty lines above. Each run returns a pandas DataFrame with one row per time step and one column per element, so everything pandas can do (select, plot, summarise, join to data) works on model output. `df.attrs["units"]` carries the units, and keyword arguments to `run` change any constant: `case.run(capacity=4000)`.

**Try it 4.**

- *Check.* Use `case.run` to find the list after 119 months when capacity is 3,900 patients a month.
- *Apply.* `case.with_init("Waiting list", 0)` returns a copy of the model starting from an empty list. With the default constants, how full is the list after 12 months, and how many patients a month are treated then?
- *Extend.* Replace the referral rate by a series that is 4,000 until month 60 and 4,200 after it, using `case.run(inputs={"Referral rate": series})` where `series` is a pandas Series indexed by time. Where does the list end?

<details><summary>Worked answer</summary>

*Check.* A net flow of 100 a month for 119 months gives 32,000 plus 11,900, or 43,900.

*Apply.* An empty list can supply only `Waiting_list / 0.5` treatments a month, so at first treatments are tiny and the list fills fast; once it holds 1,975 patients, capacity binds and the list rises by 50 a month. After 12 months it holds about 2,480 patients and treatments run at full capacity, 3,950 a month.

*Extend.* From month 60 the net flow is 250 a month instead of 50, so the list ends near 32,000 + 50 x 60 + 250 x 59 = 49,750. The run gives 49,725: Euler uses the referral rate at the start of each step, and the step from month 60 to 60.125 starts at 4,000, which costs 0.125 x 200 = 25 patients.
</details>

In [ ]:
lower = case.run(capacity=3900)
print(f"capacity 3,900: {lower['Waiting list'].iloc[-1]:,.0f}")
# with_init gives a copy of the model with a new starting value; T=12 runs
# for 12 months instead of the model's own 119
empty = case.with_init("Waiting list", 0).run(T=12)
print(f"from empty, after 12 months: {empty['Waiting list'].iloc[-1]:,.0f} "
      f"waiting, {empty['Treatments'].iloc[-1]:,.0f} treated a month")
# referral rate at these times, joined by straight lines: a step at month 60
# (@sec-package lists every run option, inputs= among them)
series = pd.Series([4000, 4000, 4200, 4200], index=[0, 60, 60.125, 119])
step_up = case.run(inputs={"Referral rate": series})
print(f"referrals up from month 60: list ends at "
      f"{step_up['Waiting list'].iloc[-1]:,.0f}")

## 5. Choosing the time step

Euler's method holds the flows still for a whole step, but in a model with feedback the flows change during the step. If the step is long compared with how fast the model responds, the result is wrong in ways that can look like real behaviour. The fastest response in our model is the minimum treatment time of 0.5 month: when the list is short, treatments are `Waiting_list / 0.5`, so the list closes any gap with its equilibrium with a time constant of half a month.

To see what that means, start the list at 9,000 and raise capacity to 6,000 a month, above the 4,000 referrals. The list should fall by 2,000 a month until it is too short to fill the clinics, at 3,000 patients, and then ease towards 2,000, where `Waiting_list / 0.5` equals the 4,000 referrals. The exact path after month 3 is $2000 + 1000\,e^{-(t-3)/0.5}$.

In [ ]:
# with_params gives a copy whose constants have new default values
drain = case.with_init("Waiting list", 9000).with_params(capacity=6000)
t = np.linspace(0, 12, 481)
exact = np.where(t < 3, 9000 - 2000 * t, 2000 + 1000 * np.exp(-(t - 3) / 0.5))

fig, ax = plt.subplots()
ax.plot(t, exact, label="Exact")
for dt in (1, 0.5, 0.125):
    run = drain.run(T=12, dt=dt)["Waiting list"]
    ax.plot(run.index, run, marker="o" if dt >= 0.5 else None, ms=4,
            label=f"Euler, dt = {dt} month")
ax.set_xlim(0, 8)
ax.set_ylim(0, 9500)
ax.set_title("The same model with three time steps")
ax.set_xlabel("Month")
ax.set_ylabel("Patients waiting")
ax.legend()
plt.show()

print("dt (month)   list at month 4, 5, 6 and 12")
# ", ".join(...) glues the four formatted numbers into one string
for dt in (1, 0.5, 0.25, 0.125, 0.0625):
    run = drain.run(T=12, dt=dt)["Waiting list"]
    print(f"{dt:>10}   " + ", ".join(f"{run.loc[m]:,.0f}"
                                      for m in (4, 5, 6, 12)))

Figure: the draining list with time steps of 1, 0.5 and 0.125 month against the exact path. The one-month step swings for ever between 1,000 and 3,000 patients; the eighth-of-a-month step is close to the exact curve.

With `dt = 1` the model treats 6,000 patients in the month after the list reaches 3,000, overshoots to 1,000, then under-treats and bounces back to 3,000, for ever. That oscillation is not in the system; it is arithmetic. With `dt = 0.5` the list jumps to 2,000 in one step. With 0.25 and less, the path bends smoothly into 2,000, and halving from 0.125 to 0.0625 changes the list at month 4 by 18 patients (the exact value is 2,135). Every step of half a month or less ends at 2,000, the true equilibrium, so a run that reports only where the model settles can hide a step that distorts the path on the way.

Two rules follow, and the rest of the book uses them:

1. Choose `dt` at most a quarter of the shortest time constant in the model. Here that is 0.5 / 4 = 0.125 month, the default in `case_spec`.
2. Test it by **halving**: run again with `dt / 2`. If anything you report changes by more than you care about, the step was too long.

**Try it 5.**

- *Check.* With `dt = 0.25`, what fraction of the remaining gap to 2,000 does one step remove once the list is below 3,000?
- *Apply.* Run the halving test on the original 119-month case. Does halving `dt` from 0.125 change the final list?
- *Extend.* For a goal-seeking flow `gap / tau`, one Euler step multiplies the gap by $1 - \Delta t / \tau$. For which values of $\Delta t / \tau$ does the run oscillate, and for which does it explode? Check with `dt = 1.25` on the draining list.

<details><summary>Worked answer</summary>

*Check.* One step removes $\Delta t / \tau$ = 0.25 / 0.5 = half of the gap, so from month 3 the list goes 3,000, 2,500, 2,250, 2,125 and 2,062 at month 4, as the cell below shows.

*Apply.* No. Capacity binds throughout, the net flow is a constant 50 a month, and Euler is exact for a constant net flow: both runs end at 37,950. A halving test passes trivially when nothing in the model is moving fast; it earns its keep in section 6, where the flows depend on the stock.

*Extend.* The factor $1 - \Delta t/\tau$ is negative when $\Delta t > \tau$, so the gap changes sign each step and the run oscillates. It is larger than 1 in size when $\Delta t > 2\tau$, and then the oscillation grows. With `dt = 1.25` the factor is -1.5: the run below goes 1,500, 2,750, 875, 3,688, 1,188, 3,219, the gap from 2,000 growing by half each step until the capacity of 6,000 a month caps the treatments. The swinging never dies away, and none of it is in the system.
</details>

In [ ]:
r = drain.run(T=12, dt=0.25)["Waiting list"]
# r.loc[3:4] keeps the times from 3 to 4 inclusive; tolist() prints a plain list
print("dt 0.25, months 3 to 4:", r.loc[3:4].round(0).tolist())
fine = case.run(dt=0.0625)["Waiting list"].iloc[-1]
print(f"halving test on the case: {df['Waiting list'].iloc[-1]:,.2f} "
      f"and {fine:,.2f}")
# warn_clips=False hides the warning printed when the floor at zero holds a stock
wild = drain.run(T=10, dt=1.25, warn_clips=False)
print("dt 1.25:", wild["Waiting list"].round(0).tolist())

## 6. Feedback, and why systems push back

In the first model, referrals arrive whatever happens to the list. That is an assumption, and probably a wrong one. A **feedback loop** is a closed chain of cause and effect: the stock changes something, which changes a flow, which changes the stock. Our model already has one, hidden in the `MIN`: when the list is short, a shorter list means fewer treatments, which slows the fall of the list. That is a **balancing loop**: it opposes change and pulls the stock towards a level. A **reinforcing loop** does the opposite: it amplifies change, as interest does to a debt.

Here is a second balancing loop that people who run waiting lists describe. When waits are long, family doctors refer fewer patients: some are managed in primary care, some go private, some do not bother. When waits fall, those patients come back. Let **Waiting time** be the list divided by the treatment rate (in months), and let referrals respond to it:

$$
\text{Referrals} = \text{Referral rate}\times\left(\frac{\text{Waiting time}}{\text{Normal waiting time}}\right)^{-s},
$$

where the `Normal waiting time` is 8 months, about the wait in 2014, and $s$, the `Referral sensitivity`, says how strongly referrals respond. With $s = 0.5$, a wait 21% shorter than normal brings about 12% more referrals. With $s = 0$ the loop is cut and referrals are constant again, so one model holds both stories. The values are illustrative, not estimated; chapter 3, From story to diagram, builds this loop from interviews, and chapter 13 asks what the data can say about it.

The policy question: add 400 treatments a month of capacity (10%) from month 12. What happens to the list? A model is just a dictionary, so we copy the case and edit it.

In [ ]:
# deepcopy copies the dictionary and every list inside it, so the edits below
# leave case_spec unchanged (a plain copy would share the inner lists)
loop_spec = copy.deepcopy(case_spec)
loop_spec["name"] = "Waiting list with referral feedback"
# flows[0] is Referrals; in equations ^ raises to a power
loop_spec["flows"][0]["eq"] = ("Referral_rate * (Waiting_time"
                               " / Normal_waiting_time)"
                               " ^ (-Referral_sensitivity)")
# vars[1] is Capacity; STEP(h, t) is 0 before time t and h from then on
loop_spec["vars"][1]["eq"] = "3950 + STEP(Extra_capacity, 12)"
loop_spec["vars"] += [                   # += adds four variables to the list
    {"id": "extra", "name": "Extra capacity", "eq": "0",
     "units": "patients/month"},
    {"id": "wt", "name": "Waiting time", "eq": "Waiting_list / Treatments",
     "units": "month"},
    {"id": "nwt", "name": "Normal waiting time", "eq": "8", "units": "month"},
    {"id": "sens", "name": "Referral sensitivity", "eq": "0.5",
     "units": "dimensionless"},
]
looped = lb.Model.from_spec(loop_spec)

runs = {
    "No extra capacity": looped.run(extra_capacity=0),
    "Extra capacity, referrals fixed (s = 0)":
        looped.run(extra_capacity=400, referral_sensitivity=0),
    "Extra capacity, referrals respond (s = 0.5)":
        looped.run(extra_capacity=400),
}
fig, ax = plt.subplots()
for label, run in runs.items():
    ax.plot(run.index, run["Waiting list"] / 1000, label=label)
ax.axvline(12, color="grey", ls=":", lw=1)
ax.set_ylim(0, 40)
ax.set_title("Adding 10% capacity, with and without the referral loop")
ax.set_xlabel("Months since January 2014")
ax.set_ylabel("Patients waiting (thousands)")
ax.legend(loc="lower left")
plt.show()

cols = ["Waiting list", "Waiting time", "Referrals", "Treatments"]
# one column per run, from a dict comprehension; .T turns columns into rows
summary = pd.DataFrame({label: run.loc[119, cols]
                        for label, run in runs.items()}).T
print("At month 119:")
print(summary.round(1).to_string())
print()
for label, run in runs.items():
    print(f"{label}: list at month 60 = {run.loc[60, 'Waiting list']:,.0f}")

Figure: the waiting list with no extra capacity, and with 400 extra treatments a month from month 12 (dotted line), when referrals are fixed and when they respond to the waiting time.

With referrals fixed, the extra capacity drains the list by 350 patients a month: 15,800 waiting by month 60 and only 2,000 by month 119, the shortest list that can keep the clinics busy. That is the answer a spreadsheet with fixed demand would give. With referrals responding, the list falls for a year or two and then stops, near 29,400. The wait has fallen from about 8.2 to 6.8 months, so about 400 more referrals arrive each month, and they use up all of the new capacity. The extra 400 treatments a month are all still happening; they are just treating patients who would otherwise not have been on the list.

This is **policy resistance**: the system has a goal of its own (a wait that referrers will tolerate), and a policy that pushes the stock without changing that goal is offset by the loop that defends it [@sterman2000; @meadows2008]. It is not the same as failure. The policy bought 400 more treatments a month and a wait 1.4 months shorter. It did not buy a short list, and a manager judged on the list alone would call it a failure. The data in section 1, where extra activity from 2022 stopped the growth but did not reverse it, is at least consistent with a loop like this; whether it is the reason is a hypothesis to test, not a finding.

**Try it 6.**

- *Check.* Without the extra capacity, does the list with referral feedback grow, fall or settle?
- *Apply.* Set `referral_sensitivity=1`. What happens to the list after the extra capacity, and why? (Hint: write Referrals x Waiting time.)
- *Extend.* With `referral_sensitivity=2`, the extra capacity makes the list *longer*. Check this, and explain what has happened to the waiting time. What does it tell you about judging a service by the length of its list?

<details><summary>Worked answer</summary>

*Check.* It settles. Starting at 32,000, referrals (3,975 a month) are slightly above capacity (3,950), so the list creeps up, which lengthens the wait, which trims referrals, until they match capacity at about 32,400. The balancing loop holds the list where referrals equal capacity.

*Apply.* With $s = 1$, Referrals x Waiting time = Referral rate x Normal waiting time = 32,000 always, and in equilibrium Waiting time = Waiting list / Treatments with Referrals = Treatments, so the list settles at exactly 32,000 whatever the capacity. The run ends at 32,000: the extra capacity shortens the wait to 7.4 months but leaves the list unchanged.

*Extend.* The list ends at about 33,400, above the 32,400 of the no-policy run. The wait is 7.7 months, shorter than before. A list is waiting time multiplied by throughput; when throughput rises more than the wait falls, the list grows. A service that treats more patients sooner can have a longer list, which is why waiting time, not list length, is the measure to report.
</details>

In [ ]:
base = runs["No extra capacity"]["Waiting list"]
print(f"no extra capacity: {base.iloc[0]:,.0f} at the start, "
      f"{base.iloc[-1]:,.0f} at month 119")
for s in (1, 2):
    r = looped.run(extra_capacity=400, referral_sensitivity=s)
    print(f"s = {s}: list {r['Waiting list'].iloc[-1]:,.0f}, "
          f"waiting time {r['Waiting time'].iloc[-1]:.2f} months, "
          f"Referrals x Waiting time = "
          f"{(r['Referrals'] * r['Waiting time']).iloc[-1]:,.0f}")

## 7. Four building blocks: bathtub, conversion, split and broken cascade

Most models are built from a small number of recurring structures. Jim Hines collected them as **molecules**: named fragments of stock-and-flow structure, each solving one modelling problem, that combine into larger models [@hines_molecules]. The `loopbricks` package ships 73 of them as runnable models; `lb.molecule(key)` loads one. Each molecule has a home chapter where it is explained; four of them belong here, because they are about how material moves between stocks. Our versions are our own implementations of the structures Hines describes, and each one notes where it differs.

### Bathtub (`bathtub`)

**What it is for:** anything that accumulates, increased by one flow and decreased by another. It is the molecule our case model is built on.

$$
\text{Level}(t) = \text{Level}(0) + \int_0^t (\text{Filling} - \text{Draining})\,ds,\qquad
\text{Draining} = \min\left(\text{Outflow},\ \frac{\text{Level}}{\text{Minimum draining time}} + \text{Inflow}\right).
$$

In [ ]:
tub = lb.molecule("bathtub")
tub_run = tub.run(preset="Draining")         # inflow 3, outflow 12 a year
fig, ax = plt.subplots()
ax.plot(tub_run.index, tub_run["Level"], label="Level (units)")
ax.plot(tub_run.index, tub_run["Draining"], label="Draining (units a year)")
ax.set_xlim(0, 10)
ax.set_title("Bathtub, Draining preset")
ax.set_xlabel("Year")
ax.set_ylabel("Units, or units a year")
ax.legend()
plt.show()
low = tub_run.index[tub_run["Level"] < 1][0]   # first time the level is below 1
print(f"Level below 1 unit from year {low}; draining at the end: "
      f"{tub_run['Draining'].iloc[-1]:.1f} units a year")

Figure: the Bathtub molecule with the Draining preset (inflow 3, outflow 12 units a year). The level falls in a straight line, then the drain slows as the tub empties.

The level falls by 9 units a year from 50 and is below 1 unit by year 5.5. Then the minimum draining time takes over and the drain slows to the 3 units a year coming in. After Hines, *Molecules of Structure* [@hines_molecules]. Hines leaves both flows open and warns that a physical stock needs an outflow that depends on the stock; ours fills that in with a minimum draining time, the same device as the minimum treatment time in the case. Ours also adds the inflow to the limit, so a tub that is empty but filling can pass its inflow straight through; the case model omits that term, so an empty list treats no one until patients arrive.

### Conversion (`conversion`)

**What it is for:** material that changes form, and units, as it moves from one stock to another: referrals that become appointments, ore that becomes metal. The flow out of the first stock and the flow into the second are linked by a conversion factor, the `Yield`:

$$
\text{Processing} = \frac{\text{Raw}}{\text{Process time}},\qquad
\text{Production} = \text{Processing}\times\text{Yield}.
$$

In [ ]:
conv = lb.molecule("conversion")
conv_run = conv.run()                        # yield 2, process time 4 months
fig, ax = plt.subplots()
ax.plot(conv_run.index, conv_run["Raw"], label="Raw")
ax.plot(conv_run.index, conv_run["Finished"], label="Finished")
ax.set_title("Conversion: two units of output for each unit of input")
ax.set_xlabel("Month")
ax.set_ylabel("Stock (units of each kind)")
ax.legend()
plt.show()
print(conv_run.iloc[-1][["Raw", "Finished", "Processing",
                         "Production"]].round(1).to_string())

Figure: the Conversion molecule with its default settings (feed 10 a month, process time 4 months, yield 2).

In equilibrium 10 raw units a month are processed and become 20 finished units a month, so the finished stock settles at 120 (20 a month for the 6 months they stay). In the waiting list, think of each new patient seen as generating, on average, a number of review appointments: the yield converts patients into appointments, and the review list is a stock in different units. After Hines, *Molecules of Structure* [@hines_molecules]. Hines's Conversion moves people from one category to another with a single flow (non-customer to customer); his Broken Cascade covers a change of units. Ours combines the two ideas: a broken chain with a yield, so the units change on the way.

### Split Flow (`split`)

**What it is for:** one flow divided between destinations by a fraction: urgent and routine referrals, public and private patients, market shares.

$$
\text{To A} = \text{Inflow}\times f,\qquad \text{To B} = \text{Inflow}\times(1 - f),
$$

where $f$ is the `Fraction to A`, a number between 0 and 1.

In [ ]:
split = lb.molecule("split")
split_run = split.run()                      # inflow 14 a month, 60% to A
fig, ax = plt.subplots()
ax.plot(split_run.index, split_run["Channel A"], label="Channel A (60%)")
ax.plot(split_run.index, split_run["Channel B"], label="Channel B (40%)")
ax.set_title("Split Flow: one inflow, two destinations")
ax.set_xlabel("Month")
ax.set_ylabel("Stock (units)")
ax.legend()
plt.show()
end = split_run.iloc[-1]
print(f"Channel A {end['Channel A']:.1f}, Channel B {end['Channel B']:.1f}, "
      f"total {end['Channel A'] + end['Channel B']:.1f}")

Figure: the Split Flow molecule with its default settings (inflow 14 a month, 60% to channel A, drain time 5 months).

Each channel settles at its share of the inflow times the drain time: 8.4 x 5 = 42 in A and 5.6 x 5 = 28 in B. Whatever the fraction, the two flows add up to the inflow, so nothing is created or lost by the split. In the case, triage splits referrals into urgent and routine, and the two streams then wait in different queues. After Hines, *Molecules of Structure* [@hines_molecules]. Hines splits the outflow of a source stock into two sub-flows; ours splits an inflow from outside the model and gives each destination its own drain, so a single run shows where the material ends up.

### Broken Cascade (`broken`)

**What it is for:** a chain of stocks where material can leave part-way instead of always moving on: patients removed from a list before they are seen, students who drop out, sales leads that go cold.

$$
\text{Exiting stage 1} = \frac{\text{Stage 1}}{\text{Flow time}},\quad
\text{Advance} = \text{Exiting stage 1}\times(1 - \ell),\quad
\text{Leaking} = \text{Exiting stage 1}\times\ell,
$$

where $\ell$ is the `Leak fraction`.

In [ ]:
broken = lb.molecule("broken")
dt_b = broken.dt                       # the model's own time step
print("leak fraction   leaked   advanced   (of 80 units in stage 1)")
for preset in ("Low leak", None, "High leak"):
    run = broken.run(preset=preset) if preset else broken.run()   # None: defaults
    leaked = run["Leaking"].iloc[:-1].sum() * dt_b      # add up the flow
    advanced = run["Advance"].iloc[:-1].sum() * dt_b
    frac = run["Leak fraction"].iloc[0]
    print(f"{frac:>13}   {leaked:6.1f}   {advanced:8.1f}")
broken_run = broken.run()
fig, ax = plt.subplots()
ax.plot(broken_run.index, broken_run["Stage 1"], label="Stage 1")
ax.plot(broken_run.index, broken_run["Stage 2"], label="Stage 2")
ax.set_xlim(0, 30)
ax.set_title("Broken Cascade, leak fraction 0.3")
ax.set_xlabel("Month")
ax.set_ylabel("Stock (units)")
ax.legend()
plt.show()

Figure: the Broken Cascade molecule with its default settings: 80 units start in stage 1, 30% of those leaving it leak away, the rest pass to stage 2.

Of the 80 units, 24 leak and 56 reach stage 2 with the default leak fraction of 0.3; with 0.1 and 0.8 the split is 8 and 72, or 64 and 16. (The totals add up the flow over the run: the sum of a flow times `dt` is its integral.) In the case, a validation exercise that writes to patients and removes those who no longer need an appointment is a leak: the list falls, but no one has been treated. After Hines, *Molecules of Structure* [@hines_molecules]. Hines's Broken Cascade ends the first stock's outflow in a cloud and starts the next stock's inflow from it, usually with a change of units; ours keeps the units and sends a fixed fraction of the outflow out of the chain, which is the use the case needs.

**Try it 7.**

- *Check.* In the Split Flow molecule, what fraction to A makes channel A hold 56 units in the long run?
- *Apply.* Which molecule would you use for each of these, and why: (a) a pharmacy's stock of a drug; (b) patients referred to either a public or a private clinic; (c) job applicants, some of whom withdraw before interview?
- *Extend.* Run the Conversion molecule with the High yield and Low yield presets. Where does the finished stock settle in each, and what single number explains the difference?

<details><summary>Worked answer</summary>

*Check.* Channel A settles at 14 x f x 5 = 70 f, so 56 needs f = 0.8. The cell below confirms it.

*Apply.* (a) Bathtub: one stock, deliveries in, dispensing out. (b) Split Flow: one flow of referrals divided by a fraction. (c) Broken Cascade: applicants move from applied to interviewed, and some leave part-way.

*Extend.* With yield 4 the finished stock settles at 240; with 0.6 at 36. Raw input is 10 a month in both, so production is 10 x yield a month, and the finished stock is production times the 6 months it stays: 60 x yield.
</details>

In [ ]:
print(f"fraction 0.8: channel A settles at "
      f"{split.run(fraction_to_a=0.8)['Channel A'].iloc[-1]:.1f}")
for preset in ("High yield", "Low yield"):
    run = conv.run(preset=preset)
    print(f"{preset}: finished stock {run['Finished'].iloc[-1]:.1f}, "
          f"yield {run['Yield'].iloc[0]}")

## 8. Why Python for system dynamics

Most system dynamics is still done in graphical tools, and they are good at what they do. This book uses Python for four reasons that matter to an analyst whose work must stand up in front of decision makers.

- **Reproducibility.** A notebook records every step from raw data to the final chart. Anyone can rerun it and get the same numbers, and you can see exactly what changed between two versions of a model with ordinary version control.
- **Data.** Real projects start from messy data. Python's pandas, numpy and matplotlib let you clean the data, run the model and compare the two in the same place, as section 2 did with the net flow.
- **Testing.** A model is code, so it can be tested like code. The cell below checks a property every stock-and-flow model must have: patients are neither created nor lost, so the final list equals the starting list plus everything that flowed in minus everything that flowed out. Chapter 12, Testing your model, builds a full test suite.
- **Sharing.** Models in Python can be run by thousands of runs at once for uncertainty analysis, wrapped in an interactive tool, or linked to machine learning, which chapters 13 to 15 do. Graphical tools can be connected to Python too; @sec-vensim-stella shows how to bring in Vensim and Stella models with PySD [@pysd2022].

In [ ]:
def check_patients_conserved(run):
    """Final list = first list + referrals - treatments, over the run."""
    dt = run.index[1] - run.index[0]
    referred = run["Referrals"].iloc[:-1].sum() * dt
    treated = run["Treatments"].iloc[:-1].sum() * dt
    expected = run["Waiting list"].iloc[0] + referred - treated
    error = abs(run["Waiting list"].iloc[-1] - expected)
    assert error < 1e-6, f"{error} patients unaccounted for"
    return referred, treated


for label, run in runs.items():
    referred, treated = check_patients_conserved(run)
    print(f"{label}: {referred:,.0f} referred, {treated:,.0f} treated, "
          f"balance holds")

The test passes for all three policy runs: over ten years, every patient referred is either treated or still on the list. If a future edit breaks that (a floor that clips the stock, a flow attached to the wrong stock), the `assert` fails loudly instead of letting a wrong number reach a report.

## 9. How this book is organised

The book follows one running case, this waiting list, from a diagram to a tested, calibrated model and a policy recommendation. It is organised by the stages of a modelling project, and each stage ends with a chapter on how AI helps at that stage and how to check it.

| Part | Chapters | What you learn |
|---|---|---|
| I Foundations | 1 Why systems push back; 2 Modelling with AI partners | Stocks, flows, feedback; what language models, reasoning models and agents can and cannot do |
| II Stage 1. Conceptualise | 3 From story to diagram; 4 AI for the problem and the hypothesis | Turning interviews and data into a diagram, a reference mode and a dynamic hypothesis |
| III Stage 2. Formulate | 5 Loops that grow, settle and tip; 6 Delays, pipelines and memory; 7 Decisions inside the model; 8 Why things swing; 9 Patterns that repeat; 10 AI for formulation | The structures behind growth, delay, decision rules, oscillation and recurring traps, written as equations |
| IV Stage 3. Test | 11 Learning from data; 12 Testing your model; 13 Data, calibration and uncertainty; 14 AI for testing and calibration | Building confidence, fitting to data, saying how sure you are |
| V Stage 4. Decide | 15 From model to decision; 16 AI for policy and decisions | Policy design, robustness and communication |
| VI Applications | 17 to 20: a supply chain, an epidemic, a project, a fishery | The whole process, start to finish, in other fields |

The appendices cover Python, the package, the molecule catalogue, a glossary, Vensim and Stella, and a toolkit for a thesis or project.

## Check it

Three habits catch most mistakes in a new model. Run them every time.

1. **Units.** Every flow into or out of a stock must be in stock units per time unit. `df.attrs["units"]` lists them.
2. **Extreme conditions.** Push a constant to an extreme and ask whether the model still makes physical sense: no capacity should make the list grow by every referral; no referrals should empty the list without it going negative.
3. **Halve `dt`.** Rerun with half the step; the numbers you report should not move.

In [ ]:
print("Units:", case.run(T=1).attrs["units"])
no_cap = case.run(capacity=0)["Waiting list"].iloc[-1]
print(f"capacity 0: list ends at {no_cap:,.0f} (32,000 + 4,000 x 119 = "
      f"{32000 + 4000 * 119:,})")
no_ref = case.run(referral_rate=0)
# attrs["clipped"] names each stock the floor at zero had to hold (empty if none)
print(f"no referrals: lowest list {no_ref['Waiting list'].min():.2f}, "
      f"floor needed: {bool(no_ref.attrs['clipped'])}")
coarse = looped.run(extra_capacity=400)["Waiting list"].iloc[-1]
fine = looped.run(extra_capacity=400, dt=0.0625)["Waiting list"].iloc[-1]
print(f"halving dt in the feedback model: {coarse:,.1f} and {fine:,.1f}")

All three pass. With no capacity the list gains all 4,000 referrals a month. With no referrals it drains towards zero through the minimum treatment time and never needs the floor at zero, which shows the `MIN` in the treatment rule is doing its job. Halving the step changes the policy run by less than one patient in 29,000.

## Running case

This chapter's increment is the waiting list as a bathtub: one stock, `Waiting list` (patients), with the flows `Referrals` and `Treatments` (patients a month), where `Treatments = MIN(Capacity, Waiting_list / Minimum_treatment_time)`, written once in plain Python and once as `case_spec` for `loopbricks`. A first feedback loop, referrals that respond to the waiting time, turned a policy that looked like it would empty the list into one that shortens the wait. Chapter 3 asks where loops like that come from, and finds several more.

In [ ]:
print(looped.equations())

## Tell the decision maker

Adding 10% capacity (400 treatments a month) will help patients, but it will probably not shrink the list much. If shorter waits draw back patients who had stopped being referred, the model says the list ends up only about 3,000 shorter than without the policy, while waits fall from about 8.2 to 6.8 months and 400 more patients a month are treated. Judge the policy by waiting time and patients treated, not by the length of the list. How strongly referrals respond to waits is unknown; measuring it is the most valuable next step.

## Going further

- **Other integration methods.** `run(method="rk4")` uses fourth-order Runge-Kutta, which is more accurate per step for smooth models. Euler with a tested step is the default here because models with `MIN`, `IF` and `STEP` are not smooth, and because it matches the LoopBricks app exactly.
- **Many runs at once.** `looped.run_batch(settings)` runs one simulation per row of a DataFrame of settings, vectorised with numpy; chapter 13 uses it for uncertainty.
- **Other Python tools.** PySD runs Vensim and XMILE models in Python [@pysd2022]; @sec-vensim-stella shows how. A model in `loopbricks` form is plain JSON, so it can also be checked or generated by other code.

## Mini-project

Extend the running case with a second outflow, **Removals**: patients who leave the list without treatment (they recover, go private or are removed at validation). Model it as the list divided by an average time to removal of 60 months, so removals depend on the list.

1. Add it to `case_spec` and draw the new stock-and-flow diagram by hand or with `draw_stock_flow`.
2. Recheck conservation with a version of `check_patients_conserved` that includes the new flow.
3. Rerun the capacity policy of section 6 with and without the referral loop. Which of the two balancing loops (removals or referrals) does more to hold the list in place?
4. Write three sentences for a manager on why a fall in the list is not the same as patients being treated.

## Further reading

- @meadows2008 is the most readable introduction to stocks, flows and feedback.
- @sterman2000, chapters 1 and 6, treats policy resistance and the mathematics of accumulation in depth.
- @forrester1961 is where the stock-and-flow method began, written about firms rather than hospitals.
- @homer2006 surveys what system dynamics has offered public health and where it could do more.
- @hines_molecules is the source of the molecules used throughout this book.

To play with the Bathtub molecule in a browser, open it in the optional [LoopBricks app](https://waelrash1.github.io/loopbricks/#/m/bathtub).